# LeetCode 316 — Remove Duplicate Letters bằng Python

Link đề: https://leetcode.com/problems/remove-duplicate-letters/ · Độ khó: **Medium** · Chủ đề: String, Stack, Greedy, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def removeDuplicateLetters(self, s: str) -> str:
```

(Bài này giống hệt LeetCode 1081: Smallest Subsequence of Distinct Characters.)

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ chuỗi `s`, xóa bớt ký tự sao cho **mỗi chữ cái xuất hiện đúng một lần** và kết quả là chuỗi **nhỏ nhất theo thứ tự từ điển** trong mọi khả năng. Kết quả luôn là một **dãy con (subsequence)** của `s` (giữ nguyên thứ tự tương đối).
2. Đi từ **vét cạn hoán vị các chữ cái** → **tham lam đệ quy (chọn chữ cái nhỏ nhất mà phần đuôi vẫn đủ chữ)** → **ngăn xếp đơn điệu với vị trí xuất hiện cuối** → **ngăn xếp đơn điệu với đếm số lần còn lại** → **tham lam lặp dùng mặt nạ bit (bitmask) của các chữ cái ở phần đuôi**.
3. Hiểu **ngăn xếp đơn điệu (monotonic stack)**: giữ ngăn xếp có thứ tự tăng, và **chỉ bỏ phần tử đỉnh khi chắc chắn nó còn xuất hiện lại phía sau**.
4. Biết phương pháp kiểm thử bài tham lam: đối chiếu với **brute force** (thử mọi hoán vị của các chữ cái khác nhau, lấy hoán vị nhỏ nhất là dãy con của `s`) trên chuỗi ngẫu nhiên có seed, cộng thêm các **tính chất** (mỗi chữ cái đúng một lần, là dãy con, idempotent).
5. Tránh lỗi kinh điển: chỉ giữ lần xuất hiện đầu (`dict.fromkeys`), sắp xếp bảng chữ cái, quên kiểm tra "đã có trong ngăn xếp", bỏ phần tử đỉnh mà không còn lần xuất hiện sau, và quên bỏ đánh dấu khi pop.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, hãy xóa các chữ cái trùng lặp sao cho **mỗi chữ cái xuất hiện đúng một lần**. Kết quả phải là chuỗi **nhỏ nhất theo thứ tự từ điển** trong tất cả các kết quả có thể.

**Ví dụ 1:** `s = "bcabc"` → `"abc"`.

**Ví dụ 2:** `s = "cbacdcbc"` → `"acdb"`.

**Ràng buộc:** `1 <= s.length <= 10⁴`; `s` chỉ gồm các chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random
from itertools import permutations, combinations

examples = [
    ("bcabc", "abc"),
    ("cbacdcbc", "acdb"),
]
for s, e in examples:
    print("s =", repr(s), "-> kỳ vọng", repr(e))

s = 'bcabc' -> kỳ vọng 'abc'
s = 'cbacdcbc' -> kỳ vọng 'acdb'


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và so sánh từ điển

`t` là dãy con của `s` nếu xóa vài ký tự của `s` (giữ nguyên thứ tự còn lại) ra `t`. So sánh từ điển: so từng ký tự từ trái sang phải; ký tự khác nhau đầu tiên quyết định; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn. Ở bài này mọi kết quả hợp lệ có **cùng độ dài** (số chữ cái khác nhau), nên chỉ cần so ký tự khác nhau đầu tiên.

In [2]:
def is_subsequence(t, s):
    it = iter(s)
    return all(ch in it for ch in t)          # mỗi ký tự của t phải xuất hiện (lần lượt) trong phần còn lại của s


print(is_subsequence("abc", "bcabc"), is_subsequence("acb", "bcabc"), is_subsequence("cab", "bcabc"))
print("abc" < "acb" < "bac", "ab" < "abc")

True False True
True True


### 3.2. Vị trí xuất hiện cuối cùng của từng chữ cái

`last[c]` = chỉ số lớn nhất mà `c` xuất hiện trong `s`. Nếu đang ở vị trí `i < last[c]` thì `c` **còn xuất hiện lại** phía sau; nếu `i == last[c]` thì đây là **cơ hội cuối** để dùng `c`.

In [3]:
s = "cbacdcbc"
last = {c: i for i, c in enumerate(s)}      # lần ghi sau đè lần ghi trước => vị trí cuối
print(last)

{'c': 7, 'b': 6, 'a': 2, 'd': 4}


### 3.3. Ngăn xếp đơn điệu (monotonic stack)

Ngăn xếp mà các phần tử luôn có thứ tự (ở đây **tăng dần**). Khi thêm phần tử `x`, ta **pop** những phần tử đỉnh vi phạm thứ tự, rồi push `x`. Mỗi phần tử được push và pop tối đa một lần nên tổng chi phí `O(n)`. Bài này thêm một điều kiện: **chỉ pop nếu phần tử đó còn xuất hiện lại sau** (để không bao giờ mất chữ cái).

In [4]:
def next_smaller_like(nums):
    stack, out = [], []
    for x in nums:
        while stack and stack[-1] > x:
            stack.pop()
        stack.append(x)
        out.append(list(stack))
    return out


for state in next_smaller_like([3, 1, 4, 2]):
    print(state)

[3]
[1]
[1, 4]
[1, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`s = "cbacdcbc"`, các chữ cái khác nhau: `a, b, c, d`. Kết quả dài 4.

- Chữ cái **đầu tiên** của đáp án nên **nhỏ nhất có thể**. Ứng viên nhỏ nhất là `a` (ở chỉ số 2). Nếu chọn `a` làm chữ đầu, phần đuôi `"cdcbc"` (từ sau `a`) có đủ các chữ cái còn lại `b, c, d` không? Có. Vậy chữ đầu là `a`.
- Đuôi `"cdcbc"`, bỏ hết `a`: cần `b, c, d`. Ứng viên nhỏ nhất là `b` (ở chỉ số 6, đuôi sau nó chỉ là `"c"` thiếu `d`) → không được. Thử `c` (chỉ số đầu tiên 0 trong đuôi): đuôi từ `c` là `"cdcbc"` có đủ `b, c, d`. Chữ thứ hai là `c`.
- Còn cần `b, d` trong `"dcbc"`: ứng viên nhỏ nhất `b` (sau nó `"c"` thiếu `d`) → không; `d` → đuôi `"dcbc"` có `b`, nên chọn `d`. Cuối cùng `b`.
- Đáp án `a, c, d, b` = `"acdb"` ✓.

**Quy tắc tham lam:** chọn chữ cái **nhỏ nhất** `c` sao cho **phần đuôi từ lần xuất hiện đầu của `c`** vẫn chứa **mọi chữ cái còn cần**.

In [5]:
def greedy_by_hand(s):
    need = set(s)
    start = 0
    ans = []
    while need:
        for c in sorted(need):
            i = s.index(c, start)
            if need <= set(s[i:]):            # phần đuôi từ i vẫn đủ mọi chữ cái còn cần
                ans.append(c)
                need.remove(c)
                start = i + 1
                break
    return "".join(ans)


print(greedy_by_hand("cbacdcbc"), greedy_by_hand("bcabc"))

acdb abc


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị của các chữ cái khác nhau

**Ý tưởng.** Kết quả là một **hoán vị** của tập chữ cái khác nhau của `s` (mỗi chữ đúng một lần) và phải là dãy con của `s`. Liệt kê hoán vị theo thứ tự từ điển (hoán vị của danh sách đã sắp xếp được sinh theo thứ tự đó), hoán vị đầu tiên là dãy con của `s` chính là đáp án. Số chữ cái khác nhau `m` nên `O(m! · n)`: chỉ dùng cho `m ≤ 7`.

In [6]:
def remove_duplicate_brute(s: str) -> str:
    letters = sorted(set(s))
    for perm in permutations(letters):           # theo thứ tự từ điển
        t = "".join(perm)
        if is_subsequence(t, s):
            return t
    return ""


for s, e in examples:
    assert remove_duplicate_brute(s) == e
print("PASS ví dụ (brute force hoán vị)")

PASS ví dụ (brute force hoán vị)


**Độ phức tạp (Cách 1).** Time `O(m! · n)` với `m` là số chữ cái khác nhau (`m ≤ 26`: không khả thi); Space `O(n)`. Dùng làm **oracle** cho `m ≤ 7`.

**Nút thắt.** Ta liệt kê cả những hoán vị không phải dãy con. Thay vì thử hết, ta có thể **dựng đáp án từng ký tự** bằng tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cách nhìn tương đương:

1. **Tham lam theo vị trí đầu ra:** ký tự đầu của đáp án là chữ cái nhỏ nhất `c` sao cho đuôi `s[i:]` (với `i` là lần xuất hiện đầu của `c`) vẫn chứa mọi chữ cái. Sau đó bỏ phần trước `i` và mọi `c` còn lại rồi lặp lại cho phần còn lại. Chạy `O(26·n)`.
2. **Ngăn xếp đơn điệu (một lượt):** duyệt `s` từ trái sang phải, giữ ngăn xếp là đáp án tạm. Gặp ký tự `c`:
   - nếu `c` đã nằm trong ngăn xếp: **bỏ qua** (giữ lần xuất hiện sớm hơn tốt hơn hoặc ít nhất không tệ hơn);
   - nếu không: trong khi đỉnh ngăn xếp **lớn hơn `c`** và đỉnh đó **còn xuất hiện lại sau** (`last[đỉnh] > i`), thì **pop** đỉnh (vì thay bằng `c` nhỏ hơn đứng sớm hơn sẽ tốt hơn, và ta vẫn có thể thêm lại đỉnh sau này); rồi push `c`.

Cách 2 là phiên bản một lượt, `O(n)`, của cách 1.

In [7]:
# Đếm số bước pop/push: mỗi ký tự vào ngăn xếp tối đa một lần và ra tối đa một lần => O(n)
def stack_ops(s):
    last = {c: i for i, c in enumerate(s)}
    stack, in_stack = [], set()
    pushes = pops = 0
    for i, c in enumerate(s):
        if c in in_stack:
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:
            in_stack.discard(stack.pop())
            pops += 1
        stack.append(c)
        in_stack.add(c)
        pushes += 1
    return pushes, pops


for s in ("cbacdcbc", "bcabc", "a" * 100, "abcdefghijklmnopqrstuvwxyz" * 10):
    pu, po = stack_ops(s)
    print(f"n = {len(s):3}: push = {pu:3}, pop = {po:3} (<= n)")
    assert pu <= len(s) and po <= pu

n =   8: push =   6, pop =   2 (<= n)
n =   5: push =   5, pop =   2 (<= n)
n = 100: push =   1, pop =   0 (<= n)
n = 260: push =  26, pop =   0 (<= n)


## 7. Các cách tiếp theo

### Cách 2 — Tham lam đệ quy (cách "chọn chữ cái nhỏ nhất mà đuôi vẫn đủ")

**Giải thích.** `f(s)`: nếu `s` rỗng trả `""`. Duyệt các chữ cái khác nhau của `s` theo thứ tự tăng dần; với chữ `c`, lấy `suffix = s[s.index(c):]`; nếu `set(suffix) == set(s)` thì đáp án là `c + f(suffix[1:].replace(c, ""))`. Đây là cách phổ biến nhất của lời giải tham lam.

In [8]:
def remove_duplicate_greedy(s: str) -> str:
    if not s:
        return ""
    for c in sorted(set(s)):
        suffix = s[s.index(c):]
        if set(suffix) == set(s):                           # đuôi từ lần xuất hiện đầu của c vẫn đủ chữ cái
            return c + remove_duplicate_greedy(suffix[1:].replace(c, ""))
    return ""


for s, e in examples:
    assert remove_duplicate_greedy(s) == e
print("PASS ví dụ (tham lam đệ quy)")

PASS ví dụ (tham lam đệ quy)


**Độ phức tạp Cách 2:** Time `O(26·n)` (đệ quy sâu tối đa 26 tầng, mỗi tầng `O(26 + n)`); Space `O(n)`.

### Cách 3 — Ngăn xếp đơn điệu với `last` (cách tham khảo)

**Giải thích.** `last[c]` là chỉ số xuất hiện cuối của `c`; `vis` là tập chữ cái đang nằm trong ngăn xếp. Với mỗi `c` tại chỉ số `i`: nếu `c ∈ vis` bỏ qua; ngược lại pop các đỉnh `top > c` có `last[top] > i`, rồi push `c`.

In [9]:
def remove_duplicate_stack_last(s: str) -> str:
    last = {c: i for i, c in enumerate(s)}
    stk = []
    vis = set()
    for i, c in enumerate(s):
        if c in vis:
            continue
        while stk and stk[-1] > c and last[stk[-1]] > i:
            vis.discard(stk.pop())
        stk.append(c)
        vis.add(c)
    return "".join(stk)


for s, e in examples:
    assert remove_duplicate_stack_last(s) == e
print("PASS ví dụ (ngăn xếp + last)")

PASS ví dụ (ngăn xếp + last)


**Độ phức tạp Cách 3:** Time `O(n)`; Space `O(26)` cho `vis`/`last`, cộng `O(26)` ngăn xếp (kết quả tối đa 26 ký tự).

### Cách 4 — Ngăn xếp đơn điệu với đếm số lần còn lại

**Giải thích.** Thay vì `last`, đếm `remaining[c]` = số lần `c` còn lại **sau vị trí hiện tại**. Duyệt `s`; với `c`: giảm `remaining[c]`; nếu `c` đã trong ngăn xếp thì bỏ qua; ngược lại pop các đỉnh `top > c` có `remaining[top] > 0` (còn xuất hiện sau), rồi push `c`.

In [10]:
def remove_duplicate_stack_count(s: str) -> str:
    remaining = {}
    for c in s:
        remaining[c] = remaining.get(c, 0) + 1
    stk, in_stack = [], set()
    for c in s:
        remaining[c] -= 1                                    # c vừa được dùng: số lần còn lại phía sau
        if c in in_stack:
            continue
        while stk and stk[-1] > c and remaining[stk[-1]] > 0:
            in_stack.discard(stk.pop())
        stk.append(c)
        in_stack.add(c)
    return "".join(stk)


for s, e in examples:
    assert remove_duplicate_stack_count(s) == e
print("PASS ví dụ (ngăn xếp + đếm)")

PASS ví dụ (ngăn xếp + đếm)


**Độ phức tạp Cách 4:** Time `O(n)`; Space `O(26)`.

### Cách 5 — Tham lam lặp với mặt nạ bit của phần đuôi

**Giải thích.** Tiền xử lý `suf[i]` = mặt nạ bit các chữ cái xuất hiện trong `s[i:]` (bit `ord(c) − ord('a')`), và `first_from(c, start)` bằng `s.find(c, start)`. Mỗi vòng: duyệt chữ cái `c` từ `a` đến `z` còn cần; tìm vị trí `i = s.find(c, start)`; nếu `suf[i]` chứa mọi chữ cái còn cần (`need & ~suf[i] == 0`) thì chọn `c`, đặt `start = i + 1`, bỏ `c` khỏi `need`. Mỗi vòng `O(26)` kiểm tra, `m` vòng, mỗi `find` `O(n)`: `O(26·m + ...)`, thực tế nhanh.

In [11]:
def remove_duplicate_bitmask(s: str) -> str:
    n = len(s)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = suf[i + 1] | (1 << (ord(s[i]) - 97))
    need = suf[0]
    start = 0
    ans = []
    while need:
        for k in range(26):
            if not (need >> k) & 1:
                continue
            c = chr(97 + k)
            i = s.find(c, start)
            if need & ~suf[i] == 0:                         # đuôi từ i còn đủ mọi chữ cái cần
                ans.append(c)
                need &= ~(1 << k)
                start = i + 1
                break
    return "".join(ans)


for s, e in examples:
    assert remove_duplicate_bitmask(s) == e
print("PASS ví dụ (bitmask)")

PASS ví dụ (bitmask)


**Độ phức tạp Cách 5:** Time `O(n + m · 26 · n)` xấu nhất do `find`, thực tế gần `O(26·n)`; Space `O(n)` cho mảng `suf` (có thể rút xuống bảng 26 nếu cần).

> Cách 3 là cách nộp bài (một lượt, `O(n)`). Cách 2 và 5 là dạng tham lam trực tiếp. Cách 4 là biến thể dễ nhớ của 3. Cách 1 làm oracle.

## 8. TRACE từng bước

Theo dõi Cách 3 trên `s = "cbacdcbc"`: in ngăn xếp sau mỗi ký tự và hành động.

In [12]:
def trace_stack(s):
    last = {c: i for i, c in enumerate(s)}
    stk, vis = [], set()
    print("last =", last)
    for i, c in enumerate(s):
        if c in vis:
            print(f"i={i} c={c}: đã có trong ngăn xếp -> bỏ qua | stack = {''.join(stk)}")
            continue
        popped = []
        while stk and stk[-1] > c and last[stk[-1]] > i:
            top = stk.pop()
            vis.discard(top)
            popped.append(top)
        stk.append(c)
        vis.add(c)
        note = f"pop {popped}" if popped else "không pop"
        print(f"i={i} c={c}: {note}, push {c} | stack = {''.join(stk)}")
    return "".join(stk)


assert trace_stack("cbacdcbc") == "acdb"

last = {'c': 7, 'b': 6, 'a': 2, 'd': 4}
i=0 c=c: không pop, push c | stack = c
i=1 c=b: pop ['c'], push b | stack = b
i=2 c=a: pop ['b'], push a | stack = a
i=3 c=c: không pop, push c | stack = ac
i=4 c=d: không pop, push d | stack = acd
i=5 c=c: đã có trong ngăn xếp -> bỏ qua | stack = acd
i=6 c=b: không pop, push b | stack = acdb
i=7 c=c: đã có trong ngăn xếp -> bỏ qua | stack = acdb


Bảng tóm tắt (`last: c→7, b→6, a→2, d→4`):

| i | c | hành động | ngăn xếp |
|---|---|---|---|
| 0 | c | push | `c` |
| 1 | b | `c > b` và `last[c] = 7 > 1`: pop `c`; push `b` | `b` |
| 2 | a | `b > a` và `last[b] = 6 > 2`: pop `b`; push `a` | `a` |
| 3 | c | push | `ac` |
| 4 | d | push | `acd` |
| 5 | c | đã có: bỏ qua | `acd` |
| 6 | b | `d > b` nhưng `last[d] = 4 < 6`: **không** pop; push `b` | `acdb` |
| 7 | c | đã có: bỏ qua | `acdb` |

Ở `i = 6`, `d` không thể bị pop vì không còn `d` nào phía sau: bỏ `d` là mất chữ cái.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (ngăn xếp đơn điệu, `last` và `vis`), đúng chữ ký của LeetCode.

In [13]:
class Solution:
    def removeDuplicateLetters(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}
        stk = []
        vis = set()
        for i, c in enumerate(s):
            if c in vis:
                continue
            while stk and stk[-1] > c and last[stk[-1]] > i:
                vis.remove(stk.pop())
            stk.append(c)
            vis.add(c)
        return "".join(stk)


for s, e in examples:
    assert Solution().removeDuplicateLetters(s) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Tham lam (Cách 2, 5):** đáp án có độ dài cố định `m`, nên nhỏ nhất theo từ điển nghĩa là **chọn ký tự đầu nhỏ nhất có thể**, rồi ký tự thứ hai nhỏ nhất có thể, ... Một chữ cái `c` có thể làm ký tự đầu khi tồn tại một cách hoàn thành: lấy lần xuất hiện **đầu** của `c` (càng sớm càng chừa nhiều đuôi) và phần đuôi sau nó phải chứa mọi chữ cái còn lại. Ví dụ tại `"cbacdcbc"` chữ `a` đầu tiên ở chỉ số 2, đuôi vẫn đủ `b, c, d`.
- **Ngăn xếp (Cách 3, 4) mô phỏng tham lam:** ngăn xếp luôn là chuỗi tăng ngặt theo từng đoạn "không thể pop". Khi gặp `c` nhỏ hơn đỉnh `top` và `top` còn xuất hiện sau, ta được phép đổi: bỏ `top` ở đây (sẽ thêm lại ở lần xuất hiện sau) và đặt `c` nhỏ hơn lên đứng trước, kết quả nhỏ hơn. Nếu `top` không còn xuất hiện sau thì bắt buộc giữ.
- **Bỏ qua ký tự đã có trong ngăn xếp:** giữ lần xuất hiện sớm. Nếu nó đang nằm trong ngăn xếp thì mọi ký tự trên nó trong ngăn xếp đã lớn hơn nó (đúng thứ tự); dùng lần sau chỉ làm chuỗi lớn hơn hoặc không tốt hơn.
- **Không mất chữ cái:** chỉ pop khi chữ cái đó còn xuất hiện lại sau, và ta chỉ push khi chưa có; do đó cuối cùng mọi chữ cái đều có mặt đúng một lần.
- **Cách 4 tương đương Cách 3:** `last[top] > i` ⇔ `remaining[top] > 0` (khi đã trừ lần xuất hiện ở vị trí `i`).

## 11. Phân tích Time / Space Complexity

- **Cách 1:** Time `O(m! · n)` (`m` là số chữ cái khác nhau); Space `O(n)`.
- **Cách 2:** Time `O(26·n)`; Space `O(n)` (xâu con được sao chép ở mỗi tầng đệ quy, sâu tối đa 26).
- **Cách 3:** Time `O(n)`; Space `O(26)`.
- **Cách 4:** Time `O(n)`; Space `O(26)`.
- **Cách 5:** Time `O(n + 26·m·...)`, thực tế `O(26·n)`; Space `O(n)`.
- Với `n = 10⁴`: tất cả cách tuyến tính xử lý trong vài mili giây.

## 12. Bộ test

Mô hình kiểm thử: brute force **hoán vị** (cho số chữ cái khác nhau `≤ 7`) làm oracle. Với chuỗi lớn hoặc nhiều chữ cái: so các cách nhanh với nhau và kiểm tra bằng **tính chất** (mỗi chữ cái đúng một lần, là dãy con, có thứ tự không thể cải thiện bằng cách thử đổi chỗ hai chữ).

In [14]:
impls = {
    "brute": remove_duplicate_brute,
    "greedy": remove_duplicate_greedy,
    "stack_last": remove_duplicate_stack_last,
    "stack_count": remove_duplicate_stack_count,
    "bitmask": remove_duplicate_bitmask,
    "submit": lambda s: Solution().removeDuplicateLetters(s),
}


def valid_result(s, t):
    """Mỗi chữ cái của s đúng một lần trong t và t là dãy con của s."""
    return sorted(t) == sorted(set(s)) and is_subsequence(t, s)


def check(s, expected=None):
    distinct = len(set(s))
    base = remove_duplicate_brute(s) if distinct <= 7 else remove_duplicate_stack_last(s)
    if expected is not None:
        assert base == expected, (s, base, expected)
    assert valid_result(s, base), (s, base)
    for name, fn in impls.items():
        if name == "brute" and distinct > 7:
            continue
        got = fn(s)
        assert got == base, (name, s, got, base)
    return base


for s, e in examples:
    check(s, e)
print("PASS 2 ví dụ của đề")

PASS 2 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    ("a", "a", "một ký tự"),
    ("aaaa", "a", "toàn một chữ cái"),
    ("abcd", "abcd", "đã khác nhau và tăng dần"),
    ("dcba", "dcba", "đã khác nhau nhưng giảm dần (không thể đổi chỗ)"),
    ("abacb", "abc", "có thể chọn abc"),
    ("bbcaac", "bac", "giữ chữ cái duy nhất ở cuối"),
    ("ecbacba", "eacb", "ví dụ kinh điển"),
    ("leetcode", "letcod", "leetcode"),
    ("cdadabcc", "adbc", "cdadabcc"),
    ("bcabc", "abc", "ví dụ của đề"),
    ("zyxwvutsrqponmlkjihgfedcbaabcdefghijklmnopqrstuvwxyz", "abcdefghijklmnopqrstuvwxyz", "bảng chữ cái giảm rồi tăng: chọn bản tăng"),
    ("abcdefghijklmnopqrstuvwxyzzyxwvutsrqponmlkjihgfedcba", "abcdefghijklmnopqrstuvwxyz", "tăng rồi giảm: giữ bản tăng ở đầu"),
]
for s, e, why in cases:
    assert check(s, e) == e
    print("PASS", repr(s[:30]), "->", repr(e[:30]), "|", why)

PASS 'a' -> 'a' | một ký tự
PASS 'aaaa' -> 'a' | toàn một chữ cái
PASS 'abcd' -> 'abcd' | đã khác nhau và tăng dần
PASS 'dcba' -> 'dcba' | đã khác nhau nhưng giảm dần (không thể đổi chỗ)
PASS 'abacb' -> 'abc' | có thể chọn abc
PASS 'bbcaac' -> 'bac' | giữ chữ cái duy nhất ở cuối
PASS 'ecbacba' -> 'eacb' | ví dụ kinh điển
PASS 'leetcode' -> 'letcod' | leetcode
PASS 'cdadabcc' -> 'adbc' | cdadabcc
PASS 'bcabc' -> 'abc' | ví dụ của đề
PASS 'zyxwvutsrqponmlkjihgfedcbaabcd' -> 'abcdefghijklmnopqrstuvwxyz' | bảng chữ cái giảm rồi tăng: chọn bản tăng
PASS 'abcdefghijklmnopqrstuvwxyzzyxw' -> 'abcdefghijklmnopqrstuvwxyz' | tăng rồi giảm: giữ bản tăng ở đầu


### 12.2. Tính chất và phản ví dụ

In [16]:
rnd = random.Random(316)

# Tính chất 1: đáp án hợp lệ (mỗi chữ cái đúng một lần, là dãy con)
# Tính chất 2: idempotent f(f(s)) = f(s)
# Tính chất 3: nếu s có mọi chữ cái khác nhau thì f(s) = s
# Tính chất 4: không thể làm nhỏ hơn bằng cách hoán đổi một cặp vị trí trong đáp án (nếu hoán đổi vẫn là dãy con)
for _ in range(500):
    alphabet = "abcdefghijklmnopqrstuvwxyz"[:rnd.randint(1, 12)]
    s = "".join(rnd.choice(alphabet) for _ in range(rnd.randint(1, 60)))
    t = remove_duplicate_stack_last(s)
    assert valid_result(s, t)
    assert remove_duplicate_stack_last(t) == t
    for i in range(len(t)):
        for j in range(i + 1, len(t)):
            u = list(t)
            u[i], u[j] = u[j], u[i]
            u = "".join(u)
            if is_subsequence(u, s):
                assert u >= t
distinct = "".join(rnd.sample("abcdefghijklmnopqrstuvwxyz", 10))
assert remove_duplicate_stack_last(distinct) == distinct
print("PASS tính chất: hợp lệ, idempotent, chuỗi đã khác nhau giữ nguyên, không cải thiện được bằng hoán đổi")

# Phản ví dụ: không thể đơn giản chọn lần xuất hiện đầu tiên hay lần cuối cùng của từng chữ cái
s = "cbacdcbc"
first_occ = "".join(dict.fromkeys(s))
last_occ = "".join(c for i, c in enumerate(s) if i == max(j for j, d in enumerate(s) if d == c))
print("giữ lần đầu ->", first_occ, "| giữ lần cuối ->", last_occ, "| đúng: acdb")
assert first_occ != "acdb" and last_occ != "acdb"

PASS tính chất: hợp lệ, idempotent, chuỗi đã khác nhau giữ nguyên, không cải thiện được bằng hoán đổi
giữ lần đầu -> cbad | giữ lần cuối -> adbc | đúng: acdb


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for trial in range(1500):
    k = rnd.randint(1, 7)
    alphabet = rnd.sample("abcdefghijklmnopqrstuvwxyz", k)
    s = "".join(rnd.choice(alphabet) for _ in range(rnd.randint(1, 16)))
    check(s)

for trial in range(200):                                 # nhiều chữ cái hơn: so các cách nhanh với nhau
    k = rnd.randint(8, 26)
    alphabet = rnd.sample("abcdefghijklmnopqrstuvwxyz", k)
    s = "".join(rnd.choice(alphabet) for _ in range(rnd.randint(1, 300)))
    check(s)

for trial in range(10):                                  # n = 10^4
    k = rnd.choice([2, 5, 13, 26])
    alphabet = rnd.sample("abcdefghijklmnopqrstuvwxyz", k)
    s = "".join(rnd.choice(alphabet) for _ in range(10 ** 4))
    check(s)
check("abcdefghijklmnopqrstuvwxyz" * 385)
check("zyxwvutsrqponmlkjihgfedcba" * 385)
check("a" * 10 ** 4)
print("PASS 1500 test nhỏ (với brute), 200 test nhiều chữ cái, 10 chuỗi 10^4 ký tự ngẫu nhiên, 3 chuỗi 10^4 ký tự đặc biệt")

PASS 1500 test nhỏ (với brute), 200 test nhiều chữ cái, 10 chuỗi 10^4 ký tự ngẫu nhiên, 3 chuỗi 10^4 ký tự đặc biệt


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ giữ lần xuất hiện đầu tiên của mỗi chữ cái

In [18]:
def sai_giu_lan_dau(s):
    return "".join(dict.fromkeys(s))                    # "xóa trùng" nhưng không tối ưu thứ tự


print("giữ lần đầu ->", sai_giu_lan_dau("bcabc"), "| đúng: abc")
assert sai_giu_lan_dau("bcabc") != "abc"

giữ lần đầu -> bca | đúng: abc


**Vì sao sai:** xóa trùng kiểu này cho `"bca"` còn đáp án đúng là `"abc"` (bỏ `b, c` ở đầu, giữ `b, c` ở cuối). Mục tiêu là **nhỏ nhất từ điển**, không phải "giữ lần đầu".

### Lỗi 2: Sắp xếp bảng chữ cái

In [19]:
def sai_sap_xep(s):
    return "".join(sorted(set(s)))                      # bỏ qua ràng buộc thứ tự của chuỗi gốc


print("sắp xếp ->", sai_sap_xep("cbacdcbc"), "| đúng: acdb")
assert sai_sap_xep("cbacdcbc") != "acdb"

sắp xếp -> abcd | đúng: acdb


**Vì sao sai:** kết quả phải là **dãy con** của `s`, tức giữ thứ tự tương đối. `"abcd"` nhỏ hơn nhưng **không** là dãy con của `"cbacdcbc"`: sau `a` (chỉ số 2) thì `b, c` chỉ còn ở chỉ số 6, 7 và không có `d` nào sau đó.

### Lỗi 3: Quên kiểm tra "đã có trong ngăn xếp"

In [20]:
def sai_khong_kiem_tra_vis(s):
    last = {c: i for i, c in enumerate(s)}
    stk = []
    for i, c in enumerate(s):
        while stk and stk[-1] > c and last[stk[-1]] > i:
            stk.pop()
        stk.append(c)                                    # sai: c có thể đã nằm trong ngăn xếp (bị trùng)
    return "".join(stk)


print("không kiểm tra vis ->", sai_khong_kiem_tra_vis("cbacdcbc"), "| đúng: acdb")
assert sai_khong_kiem_tra_vis("cbacdcbc") != "acdb"

không kiểm tra vis -> acdbc | đúng: acdb


**Vì sao sai:** ký tự lặp lại (ví dụ `c` ở `i = 5` và `i = 7`) bị push lần nữa nên đáp án có trùng (`"acdbc"`). Cần `if c in vis: continue`.

### Lỗi 4: Pop mà không kiểm tra "còn xuất hiện sau"

In [21]:
def sai_pop_khong_kiem_tra(s):
    stk, vis = [], set()
    for i, c in enumerate(s):
        if c in vis:
            continue
        while stk and stk[-1] > c:                       # sai: thiếu điều kiện last[top] > i
            vis.discard(stk.pop())
        stk.append(c)
        vis.add(c)
    return "".join(stk)


print("pop vô điều kiện ->", sai_pop_khong_kiem_tra("cbacdcbc"), "| đúng: acdb")
assert sai_pop_khong_kiem_tra("cbacdcbc") != "acdb"

pop vô điều kiện -> abc | đúng: acdb


**Vì sao sai:** ở `i = 6` ký tự `b` làm pop `d` dù không còn `d` nào phía sau, nên kết quả thiếu chữ `d` (`"abc"`). Chỉ được pop khi chắc chắn chữ cái đó sẽ **xuất hiện lại**.

### Lỗi 5: Pop nhưng quên bỏ đánh dấu `vis`

In [22]:
def sai_quen_xoa_vis(s):
    last = {c: i for i, c in enumerate(s)}
    stk, vis = [], set()
    for i, c in enumerate(s):
        if c in vis:
            continue
        while stk and stk[-1] > c and last[stk[-1]] > i:
            stk.pop()                                     # sai: vis vẫn chứa chữ vừa pop
        stk.append(c)
        vis.add(c)
    return "".join(stk)


print("quên xóa vis ->", sai_quen_xoa_vis("bcabc"), "| đúng: abc")
assert sai_quen_xoa_vis("bcabc") != "abc"

quên xóa vis -> a | đúng: abc


**Vì sao sai:** sau khi `b, c` bị pop, `vis` vẫn nghĩ chúng đang trong ngăn xếp nên các lần xuất hiện sau bị bỏ qua: kết quả `"a"` mất chữ cái.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Xóa ký tự để được chuỗi nhỏ nhất/lớn nhất từ điển + ràng buộc đủ/duy nhất" → **ngăn xếp đơn điệu**: pop đỉnh lớn hơn ký tự hiện tại **nếu đỉnh còn xuất hiện lại**.
- Hai thứ phải theo dõi: `last[c]` (hoặc đếm còn lại) và `vis` (đang nằm trong ngăn xếp).
- Pop thì phải `vis.discard`; ký tự đã trong ngăn xếp thì `continue`.
- Họ hàng: **402 Remove K Digits**, **1673 Find the Most Competitive Subsequence**, **1081** (giống hệt bài này).
- Nhớ kiểm tra bằng brute force hoán vị trên bảng chữ cái nhỏ.
- Kết quả độ dài tối đa `26` nên ngăn xếp rất nhỏ; thời gian `O(n)`.

## 15. Biến thể của bài

1. **Lớn nhất theo từ điển** (đổi chiều so sánh).
2. **402 Remove K Digits:** xóa đúng `k` chữ số để số nhỏ nhất.
3. **1673 Find the Most Competitive Subsequence:** dãy con độ dài `k` nhỏ nhất từ điển.
4. **1081 Smallest Subsequence of Distinct Characters:** y hệt bài này.
5. **Chỉ kiểm tra tính hợp lệ của một đáp án** (`valid_result`).

In [23]:
def remove_duplicate_largest(s):
    """Biến thể 1: mỗi chữ cái đúng một lần, LỚN nhất theo từ điển."""
    last = {c: i for i, c in enumerate(s)}
    stk, vis = [], set()
    for i, c in enumerate(s):
        if c in vis:
            continue
        while stk and stk[-1] < c and last[stk[-1]] > i:
            vis.discard(stk.pop())
        stk.append(c)
        vis.add(c)
    return "".join(stk)


def remove_k_digits(num, k):
    """Biến thể 2 (402)."""
    stack = []
    for d in num:
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"


def most_competitive(nums, k):
    """Biến thể 3 (1673)."""
    stack = []
    n = len(nums)
    for i, x in enumerate(nums):
        while stack and stack[-1] > x and len(stack) - 1 + (n - i) >= k:
            stack.pop()
        if len(stack) < k:
            stack.append(x)
    return stack


def smallest_subsequence(s):
    """Biến thể 4 (1081)."""
    return remove_duplicate_stack_last(s)


rnd = random.Random(15)
for _ in range(300):
    k = rnd.randint(1, 6)
    s = "".join(rnd.choice("abcdef"[:k]) for _ in range(rnd.randint(1, 12)))
    largest_brute = next("".join(p) for p in permutations(sorted(set(s), reverse=True)) if is_subsequence("".join(p), s))
    assert remove_duplicate_largest(s) == largest_brute
    assert smallest_subsequence(s) == remove_duplicate_brute(s)
    digits = "".join(rnd.choice("0123456789") for _ in range(rnd.randint(1, 8)))
    kk = rnd.randint(0, len(digits))
    best = 0 if kk == len(digits) else min(int("".join(c)) for c in combinations(digits, len(digits) - kk))
    assert remove_k_digits(digits, kk) == str(best)
    arr = [rnd.randint(0, 6) for _ in range(rnd.randint(1, 9))]
    kk = rnd.randint(1, len(arr))
    assert most_competitive(arr, kk) == list(min(combinations(arr, kk)))
assert remove_k_digits("1432219", 3) == "1219"
assert remove_k_digits("10200", 1) == "200"
assert most_competitive([3, 5, 2, 6], 2) == [2, 6]
print("PASS biến thể: lớn nhất, 402, 1673, 1081")

PASS biến thể: lớn nhất, 402, 1673, 1081


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị chữ cái | thử mọi hoán vị, lấy cái đầu tiên là dãy con | `O(m!·n)` | `O(n)` | oracle `m ≤ 7` |
| 2. Tham lam đệ quy | chọn chữ nhỏ nhất mà đuôi vẫn đủ | `O(26·n)` | `O(n)` | dễ chứng minh |
| 3. Ngăn xếp + `last` | pop khi đỉnh lớn hơn và còn xuất hiện sau | `O(n)` | `O(26)` | **cách nộp bài** |
| 4. Ngăn xếp + đếm | `remaining[c]` thay `last` | `O(n)` | `O(26)` | tương đương 3 |
| 5. Tham lam + bitmask | `suf[i]` mặt nạ chữ cái đuôi | `O(26·n)` | `O(n)` | không dùng ngăn xếp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `remove_duplicate_stack_last("abacb")` và `remove_duplicate_stack_last("aaaa")`.

**Bài 2 (tính tay).** Chạy ngăn xếp (có `last`, `vis`) trên `s = "bcabc"` và viết trạng thái ngăn xếp sau mỗi ký tự.

**Bài 3 (điền TODO).** Hoàn thành `remove_duplicate_todo` theo Cách 3.

In [24]:
def remove_duplicate_todo(s):
    # TODO 1: last = {c: i for i, c in enumerate(s)}; stk = []; vis = set()
    # TODO 2: for i, c in enumerate(s): nếu c đã trong vis thì continue
    # TODO 3: while stk and stk[-1] > c and last[stk[-1]] > i: vis.discard(stk.pop())
    # TODO 4: stk.append(c); vis.add(c)
    # TODO 5: trả "".join(stk)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `remove_duplicate_rec(s)` theo Cách 2 (tham lam đệ quy) mà **không** dùng `set(suffix) == set(s)`, thay vào đó dùng mặt nạ bit hoặc đếm.

**Bài 5 (biến thể / mini challenge).** (a) Viết `remove_duplicate_largest2(s)` (lớn nhất từ điển). (b) Viết `count_valid_results(s)` đếm **số** chuỗi hợp lệ (mỗi chữ cái đúng một lần, là dãy con của `s`) cho `s` nhỏ bằng cách thử hoán vị.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: kết quả chứa mỗi chữ cái đúng một lần. Bài 2: ghi `last` trước (`b→3, c→4, a→2`).
- **Mức 2 (vừa):** Bài 1: `"abc"` và `"a"`. Bài 2: `i=0 b: [b]`; `i=1 c: [b,c]`; `i=2 a: pop c (last 4 > 2), pop b (last 3 > 2), [a]`; `i=3 b: [a,b]`; `i=4 c: [a,b,c]` ⇒ `"abc"`. Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: dùng `need = set(s)`, lấy `c` nhỏ nhất sao cho `need ⊆ set(suffix)`. Bài 5(a): đổi `<` thành `>` ở điều kiện pop (chính là `remove_duplicate_largest`). Bài 5(b): `sum(is_subsequence("".join(p), s) for p in permutations(sorted(set(s))))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert remove_duplicate_stack_last("abacb") == "abc"
assert remove_duplicate_stack_last("aaaa") == "a"

# Bài 2
assert remove_duplicate_stack_last("bcabc") == "abc"

# Bài 3
def remove_duplicate_todo_ans(s):
    last = {c: i for i, c in enumerate(s)}
    stk, vis = [], set()
    for i, c in enumerate(s):
        if c in vis:
            continue
        while stk and stk[-1] > c and last[stk[-1]] > i:
            vis.discard(stk.pop())
        stk.append(c)
        vis.add(c)
    return "".join(stk)


# Bài 4
def remove_duplicate_rec(s):
    if not s:
        return ""
    need = set(s)
    for c in sorted(need):
        i = s.index(c)
        if need <= set(s[i:]):
            return c + remove_duplicate_rec(s[i + 1:].replace(c, ""))
    return ""


# Bài 5
def remove_duplicate_largest2(s):
    return remove_duplicate_largest(s)


def count_valid_results(s):
    return sum(is_subsequence("".join(p), s) for p in permutations(sorted(set(s))))


rnd = random.Random(19)
for _ in range(300):
    k = rnd.randint(1, 6)
    s = "".join(rnd.choice("abcdef"[:k]) for _ in range(rnd.randint(1, 14)))
    base = remove_duplicate_brute(s)
    assert remove_duplicate_todo_ans(s) == base
    assert remove_duplicate_rec(s) == base
    assert remove_duplicate_largest2(s) == next("".join(p) for p in permutations(sorted(set(s), reverse=True)) if is_subsequence("".join(p), s))
    assert count_valid_results(s) >= 1
assert count_valid_results("abc") == 1 and count_valid_results("bcabc") == 4
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ngăn xếp đơn điệu với `last` và `vis`)

```python
class Solution:
    def removeDuplicateLetters(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}
        stk, vis = [], set()
        for i, c in enumerate(s):
            if c in vis:
                continue
            while stk and stk[-1] > c and last[stk[-1]] > i:
                vis.remove(stk.pop())
            stk.append(c)
            vis.add(c)
        return "".join(stk)
```

**Checklist**

- [ ] Có `last[c]` (hoặc đếm số lần còn lại) cho từng chữ cái?
- [ ] Ký tự đã có trong ngăn xếp thì `continue`?
- [ ] Chỉ pop khi đỉnh **lớn hơn** `c` **và** `last[đỉnh] > i` (còn xuất hiện lại)?
- [ ] Khi pop có bỏ đánh dấu `vis`?
- [ ] Sau vòng lặp mỗi chữ cái xuất hiện đúng một lần và là dãy con của `s`?
- [ ] Đã thử: một ký tự, toàn một chữ cái, đã tăng dần, đã giảm dần, chữ cái duy nhất ở cuối?
- [ ] Đã đối chiếu với brute force hoán vị trên input ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `removeDuplicateLetters(self, s: str) -> str`?